# 한국수출입은행 환율 데이터 Bronze 적재

한국수출입은행(exchangeJSON) API에서 환율 데이터를 조회하여 Bronze Delta 테이블에 저장하는 파이프라인입니다.

In [0]:
%python
import requests
import json
from datetime import datetime
from pyspark.sql.types import *

# ── 설정 ──────────────────────────────────────────────
# 인증키는 Databricks Secret Scope에서 관리합니다.
# Secret 생성 방법:
#   "Secret Scope 생성 (exim_scope)" 노트북을 실행하여 스코프 및 시크릿 생성
#   또는 CLI: databricks secrets put-secret --scope exim_scope --key authkey
AUTH_KEY = dbutils.secrets.get(scope="exim_scope", key="authkey")

API_URL = "https://oapi.koreaexim.go.kr/site/program/financial/exchangeJSON"
SEARCH_DATE = datetime.now().strftime("%Y%m%d")  # 오늘 날짜
DATA_TYPE = "AP01"  # 환율 데이터

# ── API 호출 ──────────────────────────────────────────
params = {
    "authkey": AUTH_KEY,
    "searchdate": SEARCH_DATE,
    "data": DATA_TYPE,
}

response = requests.get(API_URL, params=params, timeout=30)
response.raise_for_status()

raw_data = response.json()
print(f"API 응답: {len(raw_data)}개 통화 데이터 조회 완료")
print(json.dumps(raw_data[:2], indent=2, ensure_ascii=False) if raw_data else "데이터 없음")

# ── 메타데이터 추가 ──────────────────────────────────
load_ts = datetime.now()
for row in raw_data:
    row["_load_ts"] = load_ts
    row["_source"] = API_URL
    row["_search_date"] = SEARCH_DATE

# ── Spark DataFrame 생성 ─────────────────────────────
schema = StructType([
    StructField("result", IntegerType(), True),
    StructField("cur_unit", StringType(), True),
    StructField("cur_nm", StringType(), True),
    StructField("ttb", StringType(), True),
    StructField("tts", StringType(), True),
    StructField("deal_bas_r", StringType(), True),
    StructField("bkpr", StringType(), True),
    StructField("yy_efee_r", StringType(), True),
    StructField("ten_dd_efee_r", StringType(), True),
    StructField("kftc_bkpr", StringType(), True),
    StructField("kftc_deal_bas_r", StringType(), True),
    StructField("_load_ts", TimestampType(), True),
    StructField("_source", StringType(), True),
    StructField("_search_date", StringType(), True),
])

df = spark.createDataFrame(raw_data, schema=schema)
display(df)

API 응답: 23개 통화 데이터 조회 완료
[
  {
    "result": 1,
    "cur_unit": "AED",
    "ttb": "360.07",
    "tts": "367.34",
    "deal_bas_r": "363.71",
    "bkpr": "363",
    "yy_efee_r": "0",
    "ten_dd_efee_r": "0",
    "kftc_bkpr": "363",
    "kftc_deal_bas_r": "363.71",
    "cur_nm": "아랍에미리트 디르함"
  },
  {
    "result": 1,
    "cur_unit": "AUD",
    "ttb": "919.88",
    "tts": "938.47",
    "deal_bas_r": "929.18",
    "bkpr": "929",
    "yy_efee_r": "0",
    "ten_dd_efee_r": "0",
    "kftc_bkpr": "929",
    "kftc_deal_bas_r": "929.18",
    "cur_nm": "호주 달러"
  }
]


result,cur_unit,cur_nm,ttb,tts,deal_bas_r,bkpr,yy_efee_r,ten_dd_efee_r,kftc_bkpr,kftc_deal_bas_r,_load_ts,_source,_search_date
1,AED,아랍에미리트 디르함,360.07,367.34,363.71,363,0,0,363,363.71,2026-10-13T01:12:41.732Z,https://oapi.koreaexim.go.kr/site/program/financial/exchangeJSON,20261013
1,AUD,호주 달러,919.88,938.47,929.18,929,0,0,929,929.18,2026-10-13T01:12:41.732Z,https://oapi.koreaexim.go.kr/site/program/financial/exchangeJSON,20261013
1,BHD,바레인 디나르,"3,502.41","3,573.16","3,537.79","3,537",0,0,"3,537","3,537.79",2026-10-13T01:12:41.732Z,https://oapi.koreaexim.go.kr/site/program/financial/exchangeJSON,20261013
1,BND,브루나이 달러,"1,032.35","1,053.2","1,042.78","1,042",0,0,"1,042","1,042.78",2026-10-13T01:12:41.732Z,https://oapi.koreaexim.go.kr/site/program/financial/exchangeJSON,20261013
1,CAD,캐나다 달러,925.88,944.59,935.24,935,0,0,935,935.24,2026-10-13T01:12:41.732Z,https://oapi.koreaexim.go.kr/site/program/financial/exchangeJSON,20261013
1,CHF,스위스 프랑,"1,584.8","1,616.81","1,600.81","1,600",0,0,"1,600","1,600.81",2026-10-13T01:12:41.732Z,https://oapi.koreaexim.go.kr/site/program/financial/exchangeJSON,20261013
1,CNH,위안화,197.34,201.33,199.34,199,0,0,199,199.34,2026-10-13T01:12:41.732Z,https://oapi.koreaexim.go.kr/site/program/financial/exchangeJSON,20261013
1,DKK,덴마아크 크로네,197.39,201.38,199.39,199,0,0,199,199.39,2026-10-13T01:12:41.732Z,https://oapi.koreaexim.go.kr/site/program/financial/exchangeJSON,20261013
1,EUR,유로,"1,475.51","1,505.32","1,490.42","1,490",0,0,"1,490","1,490.42",2026-10-13T01:12:41.732Z,https://oapi.koreaexim.go.kr/site/program/financial/exchangeJSON,20261013
1,GBP,영국 파운드,"1,746.47","1,781.76","1,764.12","1,764",0,0,"1,764","1,764.12",2026-10-13T01:12:41.732Z,https://oapi.koreaexim.go.kr/site/program/financial/exchangeJSON,20261013


In [0]:
%python
# ── Bronze 테이블 저장 ──────────────────────────────
# 카탈로그/스키마/테이블명 설정 (필요시 수정)
CATALOG = "workspace"
SCHEMA = "bronze"
TABLE_NAME = "exchange_rate_koreaexim"

FULL_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"

# 스키마가 없으면 생성
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

# Bronze 테이블이 없으면 생성 (append 방식으로 적재)
# 매일 실행 시 기존 데이터에 추가되도록 mode="append" 사용
df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(FULL_TABLE)

print(f"✅ Bronze 테이블 저장 완료: {FULL_TABLE}")

# ── 저장 결과 확인 ────────────────────────────────────
result_df = spark.table(FULL_TABLE)
print(f"총 레코드 수: {result_df.count()}")
display(result_df.orderBy(result_df["_load_ts"].desc()))

✅ Bronze 테이블 저장 완료: workspace.bronze.exchange_rate_koreaexim
총 레코드 수: 48


result,cur_unit,cur_nm,ttb,tts,deal_bas_r,bkpr,yy_efee_r,ten_dd_efee_r,kftc_bkpr,kftc_deal_bas_r,_load_ts,_source,_search_date
1,THB,태국 바트,39.49,40.28,39.89,39,0,0,39,39.89,2026-10-12T08:41:46.969Z,https://oapi.koreaexim.go.kr/site/program/financial/exchangeJSON,20261012
1,AUD,호주 달러,925.38,944.07,934.73,934,0,0,934,934.73,2026-10-12T08:41:46.969Z,https://oapi.koreaexim.go.kr/site/program/financial/exchangeJSON,20261012
1,NZD,뉴질랜드 달러,743.57,758.6,751.09,751,0,0,751,751.09,2026-10-12T08:41:46.969Z,https://oapi.koreaexim.go.kr/site/program/financial/exchangeJSON,20261012
1,SEK,스웨덴 크로나,132.36,135.03,133.7,133,0,0,133,133.7,2026-10-12T08:41:46.969Z,https://oapi.koreaexim.go.kr/site/program/financial/exchangeJSON,20261012
1,CAD,캐나다 달러,928.83,947.6,938.22,938,0,0,938,938.22,2026-10-12T08:41:46.969Z,https://oapi.koreaexim.go.kr/site/program/financial/exchangeJSON,20261012
1,JPY(100),일본 옌,836.96,853.87,845.42,845,0,0,845,845.42,2026-10-12T08:41:46.969Z,https://oapi.koreaexim.go.kr/site/program/financial/exchangeJSON,20261012
1,EUR,유로,"1,483.44","1,513.41","1,498.43","1,498",0,0,"1,498","1,498.43",2026-10-12T08:41:46.969Z,https://oapi.koreaexim.go.kr/site/program/financial/exchangeJSON,20261012
1,NOK,노르웨이 크로네,138.55,141.34,139.95,139,0,0,139,139.95,2026-10-12T08:41:46.969Z,https://oapi.koreaexim.go.kr/site/program/financial/exchangeJSON,20261012
1,CNH,위안화,197.58,201.57,199.58,199,0,0,199,199.58,2026-10-12T08:41:46.969Z,https://oapi.koreaexim.go.kr/site/program/financial/exchangeJSON,20261012
1,GBP,영국 파운드,"1,752.93","1,788.34","1,770.64","1,770",0,0,"1,770","1,770.64",2026-10-12T08:41:46.969Z,https://oapi.koreaexim.go.kr/site/program/financial/exchangeJSON,20261012
